# Suffix direct-MH SMC tail estimation

This notebook runs only `suffix_direct_mh_smc`. Each configuration is an independent SMC run with its own terminal lambda, and each base-model tail estimate uses only that run's final weighted population. The self-normalized importance-sampling estimator is consistent, but generally biased for a finite number of particles.

In [ ]:
# 1. Mount Google Drive for persistent experiment outputs.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repository, or safely fast-forward an existing clean checkout.
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

if (REPO_ROOT / '.git').is_dir():
    dirty = subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError('The existing Colab checkout has local changes; review them before updating.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
print('Repository:', REPO_ROOT)

In [ ]:
# 3. Enter and validate the repository root.
import os
os.chdir(REPO_ROOT)
assert (REPO_ROOT / 'suffix_direct_mh_smc_experiment' / 'run_tail_estimation.py').is_file()
assert (REPO_ROOT / 'requirements-colab.txt').is_file()
print('cwd =', Path.cwd())

In [ ]:
# 4. Install the repository dependencies in this Colab runtime.
%pip install -q -r requirements-colab.txt

In [ ]:
# 5. Inspect PyTorch, CUDA, and the selected Colab accelerator.
import torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 6. EDIT PARAMETERS HERE. This is the only cell that configures the experiment.
LAMBDA_SCHEDULES = {
    'lambda_m1': '0,-0.25,-0.5,-0.75,-1',
    'lambda_m2': '0,-0.25,-0.5,-0.75,-1,-1.25,-1.5,-1.75,-2',
    'lambda_m3': (
        '0,-0.25,-0.5,-0.75,-1,-1.25,-1.5,-1.75,'
        '-2,-2.25,-2.5,-2.75,-3'
    ),
    'lambda_m4_5': (
        '0,-0.25,-0.5,-0.75,-1,-1.25,-1.5,-1.75,'
        '-2,-2.25,-2.5,-2.75,-3,-3.25,-3.5,-3.75,'
        '-4,-4.25,-4.5'
    ),
}
SEEDS = [200, 201, 202]
NUM_PARTICLES = 256
MAX_NEW_TOKENS = 100
MCMC_STEPS_PER_LEVEL = 10
ESS_THRESHOLD = 0.5

SUFFIX_MODE = 'stochastic'
SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SUFFIX_RHO_SCHEDULE = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'

BATCH_SIZE = 32
TAIL_THRESHOLDS = '6,8,10,12,15'
MIN_CORRECTED_ESS_OVER_N = 0.10
MIN_RAW_EVENT_FRACTION = 0.05
MAX_RAW_EVENT_FRACTION = 0.95
EXPERIMENT_LABEL = 'suffix_smc_terminal_lambda_sweep'
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_suffix_smc_tail_outputs')
DEVICE = 'cuda'
SKIP_COMPLETED_RUNS = True

assert len(SEEDS) == len(set(SEEDS))
assert LAMBDA_SCHEDULES
print('Configurations:', list(LAMBDA_SCHEDULES))
print('Seeds:', SEEDS)
print('Output:', OUTPUT_ROOT / EXPERIMENT_LABEL)

In [ ]:
# 7. Run each terminal-lambda configuration sequentially; particles stay batched on GPU.
import json
import sys

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('DEVICE=cuda but this Colab runtime has no CUDA GPU.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, 'suffix_direct_mh_smc_experiment/run_tail_estimation.py',
    '--device', DEVICE,
    '--seeds', ','.join(map(str, SEEDS)),
    '--num-particles', str(NUM_PARTICLES),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--lambda-schedules-json', json.dumps(LAMBDA_SCHEDULES),
    '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
    '--ess-threshold', str(ESS_THRESHOLD),
    '--suffix-mode', SUFFIX_MODE,
    '--suffix-sizes', SUFFIX_SIZES,
    '--suffix-rho-schedule', SUFFIX_RHO_SCHEDULE,
    '--batch-size', str(BATCH_SIZE),
    '--tail-thresholds', TAIL_THRESHOLDS,
    '--min-corrected-ess-over-n', str(MIN_CORRECTED_ESS_OVER_N),
    '--min-raw-event-fraction', str(MIN_RAW_EVENT_FRACTION),
    '--max-raw-event-fraction', str(MAX_RAW_EVENT_FRACTION),
    '--experiment-label', EXPERIMENT_LABEL,
    '--output-root', str(OUTPUT_ROOT),
]
if SKIP_COMPLETED_RUNS:
    command.append('--skip-completed')
subprocess.run(command, check=True)

In [ ]:
# 8. Load the tested aggregate table and display its single saved figure.
import pandas as pd
from IPython.display import Image, display

experiment_dir = OUTPUT_ROOT / EXPERIMENT_LABEL
for configuration in LAMBDA_SCHEDULES:
    for seed in SEEDS:
        run_path = experiment_dir / configuration / f'seed_{seed}' / 'run.json'
        if not run_path.is_file():
            raise FileNotFoundError(run_path)

aggregate_path = experiment_dir / 'aggregate.json'
figure_path = experiment_dir / 'terminal_lambda_sweep.png'
aggregate = json.loads(aggregate_path.read_text(encoding='utf-8'))
assert aggregate['algorithm'] == 'suffix_direct_mh_smc'
aggregate_df = pd.DataFrame(aggregate['aggregate_rows'])
display(aggregate_df)
display(Image(filename=str(figure_path)))